# 🧠 Drosophila MaleCNS Connectome PPO Training on Google Colab 🍄
Cuốn notebook này được thiết kế để huấn luyện mô hình sinh học kết nối thần kinh (Connectome Policy 30,000 nơ-ron) chơi Super Mario Bros trên **Google Colab GPU (T4 / A100)** hoàn toàn tự động, **chống 100% lỗi xung đột thư viện (NumPy 2.x, nes-py build, ROM graphics, v.v.)**.

---
### 📌 2 Cách đưa dự án lên Google Colab (chọn 1 trong 2):
* **Cách 1 (Khuyên dùng - Không lo mất file):** 
  Nén thư mục dự án trên máy tính thành `project.zip` và upload lên **Google Drive** của bạn. Chạy Cell 3 & 4 bên dưới, notebook sẽ tự động mount Drive, giải nén và lưu mọi checkpoint/log trực tiếp về Drive.
* **Cách 2 (Nhanh nhất):** 
  Nén các thư mục `src`, `data`, `checkpoints` (nếu có) thành `project.zip`. Mở Colab, bấm vào biểu tượng 📁 **Files** ở thanh bên trái và kéo thả file `project.zip` vào đó.

In [ ]:
# 1. Kiểm tra phần cứng GPU
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:    ", torch.cuda.get_device_name(0))
    print("VRAM (GB):     ", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))
    !nvidia-smi
else:
    print("⚠️ CẢNH BÁO: Bạn chưa bật GPU! Hãy vào menu: Runtime -> Change runtime type -> Chọn T4 GPU.")

In [ ]:
# 2. Cài đặt các thư viện hệ thống và Python (Khắc phục triệt để lỗi NumPy 2.x & nes-py)
print("⏳ Đang cài đặt thư viện hệ thống Linux & NES dependencies...")
!apt-get update -qq && apt-get install -y -qq ffmpeg libsm6 libxext6 libgl1-mesa-glx cmake xvfb > /dev/null 2>&1

print("⏳ Đang cài đặt các gói Python chuẩn (ghim numpy<2.0 để tránh crash gym/nes-py)...")
# Colab mặc định cài numpy 2.x khiến nes-py và gym crash. Cần hạ về numpy 1.x ổn định.
!pip install -q "setuptools<70" wheel
!pip install -q "numpy>=1.24.0,<2.0.0" "scipy>=1.10.0,<1.14.0" "pandas>=2.0.0,<2.3.0"
!pip install -q "gym-super-mario-bros==7.4.0" "nes-py==8.2.1"
!pip install -q matplotlib tqdm pyarrow

print("✅ Hoàn tất cài đặt dependencies!")

In [ ]:
# 3. Kết nối dữ liệu dự án (Google Drive hoặc Upload file Zip)
import os
import zipfile
from pathlib import Path

PROJECT_DIR = Path("/content/DeepLearningFruitFlyBrain_")

# Kiểm tra nếu chưa có mã nguồn, tìm file zip hoặc mount Drive
if not (PROJECT_DIR / "src").exists():
    colab_zip = Path("/content/project.zip")
    if colab_zip.exists():
        print(f"📦 Tìm thấy {colab_zip}, đang giải nén vào {PROJECT_DIR}...")
        with zipfile.ZipFile(colab_zip, 'r') as zip_ref:
            zip_ref.extractall("/content/DeepLearningFruitFlyBrain_")
    else:
        try:
            from google.colab import drive
            print("🔗 Đang mount Google Drive để tìm thư mục dự án...")
            drive.mount('/content/drive')
            
            drive_folder = Path("/content/drive/MyDrive/DeepLearningFruitFlyBrain_")
            drive_zip = Path("/content/drive/MyDrive/project.zip")
            
            if drive_folder.exists():
                PROJECT_DIR = drive_folder
                print(f"✅ Đã kết nối với thư mục dự án trên Google Drive: {PROJECT_DIR}")
            elif drive_zip.exists():
                print(f"📦 Tìm thấy {drive_zip} trên Drive, đang giải nén...")
                with zipfile.ZipFile(drive_zip, 'r') as zip_ref:
                    zip_ref.extractall("/content/DeepLearningFruitFlyBrain_")
                PROJECT_DIR = Path("/content/DeepLearningFruitFlyBrain_")
            else:
                print("ℹ️ Chưa tìm thấy dự án trên Drive. Bạn hãy kéo thả file 'project.zip' vào thanh Files bên trái Colab.")
        except Exception as e:
            print("Mount Drive bỏ qua hoặc chưa kích hoạt:", e)

# Nếu folder giải nén nằm lồng nhau
if not (PROJECT_DIR / "src").exists() and (PROJECT_DIR / "DeepLearningFruitFlyBrain_").exists():
    PROJECT_DIR = PROJECT_DIR / "DeepLearningFruitFlyBrain_"

if PROJECT_DIR.exists():
    os.chdir(str(PROJECT_DIR))
print(f"📂 Thư mục làm việc hiện tại: {os.getcwd()}")

In [ ]:
# 4. Kiểm tra toàn vẹn file dữ liệu Connectome & Môi trường Mario
import sys
from pathlib import Path

ROOT = Path(os.getcwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

data_dir = ROOT / "data" / "processed"
subgraph_path = data_dir / "subgraph_graph.npz"
node_map_path = data_dir / "subgraph_node_mapping.csv"
node_meta_path = data_dir / "node_metadata.csv"

missing = []
for p in [subgraph_path, node_map_path, node_meta_path]:
    if not p.exists():
        missing.append(str(p))

if missing:
    print("❌ THIẾU FILE DỮ LIỆU:")
    for m in missing:
        print("  -", m)
    print("\n👉 Bạn hãy đảm bảo đã upload thư mục 'data/processed' vào dự án!")
else:
    print("✅ Đã tìm thấy đầy đủ dữ liệu Connectome 30k nơ-ron:")
    print(f"  - Subgraph: {subgraph_path.stat().st_size / (1024*1024):.2f} MB")
    
    # Kiểm tra thử khởi tạo mô hình trên GPU
    import torch
    from src.connectome_model import ConnectomeMarioPolicy
    from src.environment import Mario2DEnv
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🚀 Thử khởi tạo ConnectomeMarioPolicy trên {device}...")
    model = ConnectomeMarioPolicy(
        subgraph_npz_path=subgraph_path,
        node_mapping_csv_path=node_map_path,
        node_metadata_csv_path=node_meta_path
    ).to(device)
    print(f"  - Mô hình khởi tạo thành công! (N={model.num_nodes:,} nơ-ron)")
    
    # Thử khởi tạo môi trường Mario
    env = Mario2DEnv(frame_stack=4, frame_skip=4)
    obs, info = env.reset()
    env.close()
    print("  - Môi trường Mario 2D hoạt động trơn tru!")
    print("🎉 TẤT CẢ ĐÃ SẴN SÀNG ĐỂ HUẤN LUYỆN!")

In [ ]:
# 5. TIẾN HÀNH HUẤN LUYỆN CONNECTOME PPO TRÊN GPU
# Tùy chỉnh tham số huấn luyện:
TOTAL_TIMESTEPS = 200000     # Số bước train thêm (ví dụ 100k, 200k, 500k)
NUM_ENVS = 4                 # Số môi trường song song (khuyên dùng 4 trên GPU T4)
RESUME = True                # Tự động tiếp tục từ checkpoint mới nhất (không bị mất bước)
MODEL_TYPE = "connectome"    # connectome hoặc baseline

# Tạo các thư mục cần thiết
Path("checkpoints").mkdir(exist_ok=True)
Path("logs").mkdir(exist_ok=True)

resume_flag = "--resume" if RESUME else ""

cmd = (
    f"python src/ppo_trainer.py "
    f"--model-type {MODEL_TYPE} "
    f"--total-timesteps {TOTAL_TIMESTEPS} "
    f"--num-envs {NUM_ENVS} "
    f"--amp "
    f"{resume_flag}"
)

print("🚀 Bắt đầu huấn luyện...")
print("Lệnh thực thi:", cmd)
!{cmd}

In [ ]:
# 6. Biểu đồ đánh giá kết quả huấn luyện (Live Dashboard)
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

csv_path = Path("logs/connectome_ppo_v2_metrics.csv")
if csv_path.exists():
    df = pd.read_csv(csv_path)
    
    fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
    
    # 1. Max X (Tiến độ vượt chướng ngại vật)
    axes[0].plot(df['total_timesteps'], df['max_x'], color='#10b981', label='Max X Position')
    axes[0].axhline(y=3161, color='red', linestyle='--', label='Cột cờ chiến thắng (X=3161)')
    axes[0].set_ylabel('Tọa độ X')
    axes[0].set_title('Tiến trình di chuyển xa nhất (Max X)')
    axes[0].legend(loc='lower right')
    axes[0].grid(True, alpha=0.3)
    
    # 2. Reward
    axes[1].plot(df['total_timesteps'], df['mean_raw_reward'], color='#3b82f6', label='Mean Raw Reward')
    axes[1].plot(df['total_timesteps'], df['mean_shaped_reward'], color='#f59e0b', label='Mean Shaped Reward')
    axes[1].set_ylabel('Điểm thưởng')
    axes[1].set_title('Điểm thưởng trung bình qua các đợt rollout')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # 3. Learning Rate & Entropy
    if 'entropy' in df.columns:
        axes[2].plot(df['total_timesteps'], df['entropy'], color='#8b5cf6', label='Entropy (Độ khám phá)')
    axes[2].set_xlabel('Tổng số timesteps')
    axes[2].set_ylabel('Chỉ số')
    axes[2].set_title('Độ khám phá hành vi (Policy Entropy)')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"📊 Thống kê mới nhất:")
    print(f"  - Tổng timesteps: {df['total_timesteps'].iloc[-1]:,}")
    print(f"  - Max X cao nhất: {df['max_x'].max():,}")
    print(f"  - Số lần chạm cờ (X=3161): {(df['max_x'] >= 3161).sum()} lần")
else:
    print("Chưa tìm thấy file log CSV!")

In [ ]:
# 7. Tải Checkpoint & Log về máy tính (1 Click Download)
import os
import zipfile
from google.colab import files

backup_archive = "connectome_trained_backup.zip"
print("📦 Đang đóng gói checkpoints và logs...")

with zipfile.ZipFile(backup_archive, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_list in os.walk("checkpoints"):
        for f in files_list:
            if f.endswith(".pt"):
                p = os.path.join(root, f)
                zipf.write(p, p)
    for root, dirs, files_list in os.walk("logs"):
        for f in files_list:
            if f.endswith(".csv"):
                p = os.path.join(root, f)
                zipf.write(p, p)

print(f"✅ Đã tạo {backup_archive} ({os.path.getsize(backup_archive)/(1024*1024):.2f} MB)")
print("⬇️ Đang tải về máy tính...")
try:
    files.download(backup_archive)
except Exception as e:
    print("Vui lòng tải file trực tiếp từ thanh Files bên trái:", backup_archive)